In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Install Essential Libraries
!pip install ultralytics
from IPython import display
display.clear_output()

In [ ]:
# Import Essential Libraries
import os
import pandas as pd
from PIL import Image
import cv2
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load a pretrained YOLOv8n model from Ultralytics
T_Model = YOLO('yolov8n.pt')
T_Model = YOLO('yolov8s-seg.pt')

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 segmentation model
model = YOLO("yolov8s-seg.pt")

# Train the model
results = model.train(
    data="/kaggle/input/yolov8-segmentation/data.yaml",
    epochs=50,
    imgsz=768,           
    batch=16,
    patience=20,
    optimizer='auto',
    name="brain_seg_yolov8"
)


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pred_dir = Path("/kaggle/working/runs/segment/predict_from_script")

# only select original prediction images (not batch or random training panels)
image_files = sorted([
    f for f in pred_dir.glob("*.*") 
    if f.suffix.lower() in [".jpg", ".png"]
    and "batch" not in f.name.lower()
    and "labels" not in f.name.lower()
    and "pred" not in f.name.lower()
])[:12]   # show 12 images

rows, cols = 3, 4
plt.figure(figsize=(4*cols, 4*rows))

for i, img_path in enumerate(image_files, start=1):
    img = mpimg.imread(str(img_path))
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
for img_path in image_files:
    img = mpimg.imread(str(img_path))
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.name)
    plt.show()


In [ ]:
!ls -R /kaggle/working

In [ ]:
!ls -R /kaggle/input


In [ ]:
from pathlib import Path

# Trained weights
WEIGHTS = Path("/kaggle/working/runs/segment/brain_seg_yolov8/weights/best.pt")

# Original test images
SRC_DIR = "/kaggle/input/yolov8-segmentation/test/images"

print("Using weights:", WEIGHTS)
print("Source images:", SRC_DIR)

# Run segmentation prediction on test set
!yolo segment predict model={WEIGHTS} source={SRC_DIR} save=True name=clean_test_pred


In [ ]:
from pathlib import Path

# Trained weights
WEIGHTS = Path("/kaggle/working/runs/segment/brain_seg_yolov8/weights/best.pt")

# Original test images
SRC_DIR = "/kaggle/input/yolov8-segmentation/test/images"

print("Using weights:", WEIGHTS)
print("Source images:", SRC_DIR)

# Run segmentation prediction on test set
!yolo segment predict model={WEIGHTS} source={SRC_DIR} save=True name=clean_test_pred


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Find the latest `clean_test_pred*` folder
segment_root = Path("/kaggle/working/runs/segment")
pred_dirs = sorted(segment_root.glob("clean_test_pred*"))
assert pred_dirs, "No 'clean_test_pred*' folder found. Run the prediction cell first."

pred_dir = pred_dirs[-1]  # take the newest one
print("Using prediction folder:", pred_dir)

# Take the first 12 jpg/png images
image_files = sorted(
    [f for f in pred_dir.glob("*.*") if f.suffix.lower() in [".jpg", ".png"]]
)[:12]

print(f"[INFO] Showing {len(image_files)} images")

rows, cols = 3, 4
plt.figure(figsize=(4 * cols, 4 * rows))

for i, img_path in enumerate(image_files, start=1):
    img = mpimg.imread(str(img_path))
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.name, fontsize=6)

plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pred_dir = Path("/kaggle/working/runs/segment/predict")
image_files = sorted(list(pred_dir.glob("*.jpg")) + list(pred_dir.glob("*.png")))[:20]

cols, rows = 5, (len(image_files)+4)//5
plt.figure(figsize=(4*cols, 4*rows))
for i, img_path in enumerate(image_files, 1):
    img = mpimg.imread(img_path)
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.axis("off")
    plt.title(img_path.name, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Install Ultralytics (if missing) + OpenCV ----
import sys, importlib, subprocess

def ensure(pkg):
    mod = pkg.split("==")[0].split("[")[0].replace("-", "_")
    try:
        importlib.import_module(mod if mod != "opencv_python_headless" else "cv2")
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("ultralytics==8.3.189")           # same version as your logs
ensure("opencv-python-headless")         # for cv2 without GUI

from ultralytics import YOLO
import cv2
print("✅ Ready:", YOLO.__name__)


In [ ]:
!pip install grad-cam
!pip install torch torchvision


In [ ]:
import cv2
import numpy as np

def show_segmentation_explainability(image_path, model, save_path="seg_xai.jpg"):
    # Run the model
    results = model(image_path)[0]
    
    # Read original image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    if results.masks is None:
        print("No masks found.")
        return
    
    masks = results.masks.data.cpu().numpy()
    h, w, _ = img.shape
    
    # Combine all masks into one
    combined_mask = np.zeros((h, w), dtype=np.float32)
    for m in masks:
        m_resized = cv2.resize(m, (w, h))
        combined_mask = np.maximum(combined_mask, m_resized)
    
    combined_mask = (combined_mask > 0.5).astype(np.float32)
    
    # Create heatmap
    heatmap = cv2.applyColorMap((combined_mask * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    # Overlay heatmap on original image
    overlay = (0.5 * heatmap + 0.5 * img).astype(np.uint8)
    
    # Save result
    cv2.imwrite(save_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    print("Explainability saved to:", save_path)


In [ ]:
!pip install YOLOv8-Explainer


In [ ]:
!pip install grad-cam


In [ ]:
!ls -R /kaggle/working/runs/segment


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

from pytorch_grad_cam import EigenCAM   # you can also use GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Your trained weights
WEIGHTS = Path("/kaggle/working/runs/segment/brain_seg_yolov8/weights/best.pt")
assert WEIGHTS.exists(), f"Weights not found: {WEIGHTS}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
# Load YOLOv8-seg model
model = YOLO(str(WEIGHTS))
inner_model = model.model.to(device)
inner_model.eval()

# Helper: get the last Conv2d layer in the network
def get_last_conv_layer(net):
    conv_layers = [m for m in net.modules() if isinstance(m, nn.Conv2d)]
    assert len(conv_layers) > 0, "No Conv2d layers found in model."
    return conv_layers[-1]

target_layer = get_last_conv_layer(inner_model)
target_layers = [target_layer]
print("Using target layer for CAM:", target_layer)


In [ ]:
# One example test image
img_dir = Path("/kaggle/input/yolov8-segmentation/test/images")
test_images = sorted(list(img_dir.glob("*.jpg")))
assert test_images, "No test images found."

img_path = test_images[0]   # you can change index
print("Explaining image:", img_path.name)

# Read and convert to RGB
bgr = cv2.imread(str(img_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
rgb_float = rgb.astype(np.float32) / 255.0

# To tensor [1, 3, H, W]
input_tensor = (
    torch.from_numpy(rgb_float)
    .permute(2, 0, 1)
    .unsqueeze(0)
    .to(device)
)


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Trained YOLOv8-seg weights
WEIGHTS = Path("/kaggle/working/runs/segment/brain_seg_yolov8/weights/best.pt")
assert WEIGHTS.exists(), f"Weights not found: {WEIGHTS}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load model
model = YOLO(str(WEIGHTS))
inner_model = model.model.to(device)
inner_model.eval()


In [ ]:
# Collect all Conv2d layers in the network
conv_layers = [m for m in inner_model.modules() if isinstance(m, nn.Conv2d)]
assert conv_layers, "No Conv2d layers found in YOLO model."

last_conv = conv_layers[-1]
print("Using last Conv2d layer for XAI:", last_conv)

# Store feature maps from the hook
feature_maps = {}

def hook_fn(module, input, output):
    # Save activation for later use
    feature_maps["value"] = output.detach()

# Register forward hook on the last conv layer
hook_handle = last_conv.register_forward_hook(hook_fn)


In [ ]:
from pathlib import Path

img_dir = Path("/kaggle/input/yolov8-segmentation/test/images")
test_images = sorted(list(img_dir.glob("*.jpg")))
assert test_images, "No test images found in test/images."

img_path = test_images[0]   # choose any index you like
print("Explaining image:", img_path.name)

# Read image in BGR then convert to RGB
bgr = cv2.imread(str(img_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


In [ ]:
# Run full YOLO prediction (normal way) – this will trigger our hook
_ = model(str(img_path), verbose=False)

# Get feature map from hooked layer
act = feature_maps.get("value", None)
assert act is not None, "Hook did not capture any activations."

# Take the first image in batch: shape [C, H, W]
act = act[0]   # tensor

print("Activation shape:", act.shape)  # e.g., [C, H, W]


In [ ]:
# Mean over channels -> [H, W]
heat = act.mean(dim=0).cpu().numpy()

# Keep only non-negative values (optional but common)
heat = np.maximum(heat, 0)

# Normalize to [0, 1]
heat -= heat.min()
if heat.max() > 0:
    heat /= heat.max()

print("Heatmap min/max:", heat.min(), heat.max())


In [ ]:
# Resize heatmap to original image size
H, W, _ = rgb.shape
heat_resized = cv2.resize(heat, (W, H))

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Original MRI")
plt.imshow(rgb)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("XAI Heatmap (Last Conv Activations)")
plt.imshow(rgb)
plt.imshow(heat_resized, cmap="jet", alpha=0.4)
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
for img_path in test_images[:6]:   # explain first 6 images
    print("Explaining:", img_path.name)
    
    bgr = cv2.imread(str(img_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Run YOLO to trigger hook
    _ = model(str(img_path), verbose=False)

    act = feature_maps.get("value", None)
    if act is None:
        print("No activation captured, skipping.")
        continue

    act = act[0]
    heat = act.mean(dim=0).cpu().numpy()
    heat = np.maximum(heat, 0)
    heat -= heat.min()
    if heat.max() > 0:
        heat /= heat.max()

    H, W, _ = rgb.shape
    heat_resized = cv2.resize(heat, (W, H))

    plt.figure(figsize=(8, 4))
    plt.suptitle(img_path.name, fontsize=8)

    plt.subplot(1, 2, 1)
    plt.imshow(rgb); plt.axis("off"); plt.title("Original")

    plt.subplot(1, 2, 2)
    plt.imshow(rgb)
    plt.imshow(heat_resized, cmap="jet", alpha=0.4)
    plt.axis("off"); plt.title("XAI Heatmap")

    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd

results_path = "/kaggle/working/runs/segment/brain_seg_yolov8/results.csv"

df = pd.read_csv(results_path)

print(df.head())


In [ ]:
# show all columns
print(df.columns)

# take last epoch
last = df.iloc[-1]

precision = last.get("metrics/precision(B)")
recall    = last.get("metrics/recall(B)")
f1        = last.get("metrics/f1(B)")

print("Precision (P):", precision)
print("Sensitivity (S):", recall)
print("F1-score (F):", f1)


In [ ]:
# XAI + FSP + per-class metrics + bar plot in one cell

from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# -----------------------------
# 1) Paths and basic setup
# -----------------------------
WEIGHTS = Path("/kaggle/working/runs/segment/brain_seg_yolov8/weights/best.pt")
RESULTS_CSV = Path("/kaggle/working/runs/segment/brain_seg_yolov8/results.csv")
DATA_YAML = "/kaggle/input/yolov8-segmentation/data.yaml"
TEST_IMG_DIR = Path("/kaggle/input/yolov8-segmentation/test/images")

assert WEIGHTS.exists(), f"Weights not found: {WEIGHTS}"
assert RESULTS_CSV.exists(), f"results.csv not found: {RESULTS_CSV}"
assert TEST_IMG_DIR.exists(), f"Test image dir not found: {TEST_IMG_DIR}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# 2) Global F, S, P from results.csv
# -----------------------------
df = pd.read_csv(RESULTS_CSV)
last = df.iloc[-1]  # last epoch

P_global = float(last["metrics/precision(B)"])  # Precision (boxes)
S_global = float(last["metrics/recall(B)"])     # Recall/Sensitivity (boxes)
F_global = 2 * P_global * S_global / (P_global + S_global) if (P_global + S_global) > 0 else 0.0

print("\n=== Global FSP (overall, from results.csv) ===")
print(f"Precision (P):   {P_global:.4f}")
print(f"Sensitivity (S): {S_global:.4f}")
print(f"F1-score (F):    {F_global:.4f}")

# -----------------------------
# 3) Per-class F, S, P from model.val()
# -----------------------------
print("\nRunning model.val(...) to get per-class metrics (may take some time)...")
model = YOLO(str(WEIGHTS))
metrics = model.val(data=DATA_YAML, split="test", verbose=False)  # if split='test' fails, change to 'val'

names = model.names  # dict: class_id -> class_name
per_class_rows = []

# According to Ultralytics metrics docs, SegmentMetrics.class_result(i)
# returns [p_box, r_box, ap50_box, ap_box, p_mask, r_mask, ap50_mask, ap_mask]
# We use the mask precision/recall for segmentation.
for cid, cname in names.items():
    cr = metrics.class_result(cid)  # list of floats
    # Fallback: if seg metrics not there for some reason, use box p/r
    if len(cr) >= 6:
        p_mask, r_mask = float(cr[4]), float(cr[5])
        ap50_m, ap_m = float(cr[6]), float(cr[7])
    else:
        p_mask, r_mask = float(cr[0]), float(cr[1])
        ap50_m, ap_m = float(cr[2]), float(cr[3])

    F_mask = 2 * p_mask * r_mask / (p_mask + r_mask) if (p_mask + r_mask) > 0 else 0.0

    per_class_rows.append(
        dict(
            class_id=cid,
            class_name=cname,
            precision=p_mask,
            sensitivity=r_mask,
            f1=F_mask,
            map50=ap50_m,
            map=ap_m,
        )
    )

df_classes = pd.DataFrame(per_class_rows)
print("\n=== Per-class FSP (mask-based) ===")
print(df_classes[["class_id", "class_name", "precision", "sensitivity", "f1"]])

# -----------------------------
# 4) Plot global F / S / P bar chart
# -----------------------------
plt.figure(figsize=(5, 4))
metrics_names = ["Precision (P)", "Sensitivity (S)", "F1-score (F)"]
metric_values = [P_global, S_global, F_global]

plt.bar(metrics_names, metric_values)
plt.ylim(0, 1.0)
for i, v in enumerate(metric_values):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
plt.ylabel("Score")
plt.title("Global F / S / P (YOLOv8 Segmentation)")
plt.tight_layout()
plt.show()

# -----------------------------
# 5) Simple XAI: last Conv activation heatmap on one test MRI
# -----------------------------
# Load inner YOLO model
inner_model = model.model.to(device)
inner_model.eval()

# Find last Conv2d layer
conv_layers = [m for m in inner_model.modules() if isinstance(m, nn.Conv2d)]
assert conv_layers, "No Conv2d layers found in YOLO model."
last_conv = conv_layers[-1]
print("\nUsing last Conv2d layer for XAI:", last_conv)

# Hook to capture feature maps
feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["value"] = output.detach()

hook_handle = last_conv.register_forward_hook(hook_fn)

# Pick one test image
test_images = sorted(TEST_IMG_DIR.glob("*.jpg"))
assert test_images, f"No .jpg images found in {TEST_IMG_DIR}"
img_path = test_images[0]
print("Explaining image:", img_path.name)

# Read image (BGR -> RGB)
bgr = cv2.imread(str(img_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# Run YOLO prediction (this triggers the hook)
_ = model(str(img_path), verbose=False)

# Get feature map from hooked layer
act = feature_maps.get("value", None)
assert act is not None, "Hook did not capture any activations."

# Use first item in batch: shape [C, H, W]
act = act[0]  # tensor
print("Activation shape:", act.shape)

# Build heatmap: mean across channels -> [H, W]
heat = act.mean(dim=0).cpu().numpy()
heat = np.maximum(heat, 0)
heat -= heat.min()
if heat.max() > 0:
    heat /= heat.max()

# Resize heatmap to image size
H, W, _ = rgb.shape
heat_resized = cv2.resize(heat, (W, H))

# Visualize original vs heatmap overlay
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.title("Original MRI")
plt.imshow(rgb)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("XAI Heatmap (Last Conv Activations)")
plt.imshow(rgb)
plt.imshow(heat_resized, cmap="jet", alpha=0.4)
plt.axis("off")

plt.tight_layout()
plt.show()

# Remove hook to avoid side effects later
hook_handle.remove()


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import shutil
import os

# -----------------------------
# Paths
# -----------------------------
RUN_DIR = Path("/kaggle/working/runs/segment/brain_seg_yolov8")
WEIGHTS = RUN_DIR / "weights/best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
DATA_YAML = "/kaggle/input/yolov8-segmentation/data.yaml"
TEST_IMG_DIR = Path("/kaggle/input/yolov8-segmentation/test/images")
RESULTS_DIR = Path("/kaggle/working/results")

assert WEIGHTS.exists(), f"Weights not found: {WEIGHTS}"
assert RESULTS_CSV.exists(), f"results.csv not found: {RESULTS_CSV}"
assert TEST_IMG_DIR.exists(), f"Test image dir not found: {TEST_IMG_DIR}"

RESULTS_DIR.mkdir(exist_ok=True, parents=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# Global F, S, P from results.csv
# -----------------------------
df = pd.read_csv(RESULTS_CSV)
last = df.iloc[-1]  # last epoch

P_global = float(last["metrics/precision(B)"])  # precision
S_global = float(last["metrics/recall(B)"])     # sensitivity/recall
F_global = 2 * P_global * S_global / (P_global + S_global) if (P_global + S_global) > 0 else 0.0

print("\n=== Global FSP (from results.csv) ===")
print(f"Precision (P):   {P_global:.4f}")
print(f"Sensitivity (S): {S_global:.4f}")
print(f"F1-score (F):    {F_global:.4f}")

# -----------------------------
# Per-class F, S, P from model.val()
# -----------------------------
print("\nRunning model.val(...) to get per-class metrics (may take some time)...")
model = YOLO(str(WEIGHTS))
metrics = model.val(data=DATA_YAML, split="test", verbose=False)  # if split=test fails, change to split='val'

names = model.names  # dict: class_id -> class_name
per_class_rows = []

# NOTE: structure of class_result() can vary; this is my *best-informed* guess
# If it errors, print(metrics.class_result(0)) and adjust indices.
for cid, cname in names.items():
    cr = metrics.class_result(cid)  # list of floats
    # Typical order: [p_box, r_box, ap50_box, ap_box, p_mask, r_mask, ap50_mask, ap_mask]
    if len(cr) >= 6:
        p_mask, r_mask = float(cr[4]), float(cr[5])
        ap50_m = float(cr[6]) if len(cr) > 6 else np.nan
        ap_m   = float(cr[7]) if len(cr) > 7 else np.nan
    else:
        # fallback: use box metrics if seg not present
        p_mask, r_mask = float(cr[0]), float(cr[1])
        ap50_m = float(cr[2]) if len(cr) > 2 else np.nan
        ap_m   = float(cr[3]) if len(cr) > 3 else np.nan

    F_mask = 2 * p_mask * r_mask / (p_mask + r_mask) if (p_mask + r_mask) > 0 else 0.0

    per_class_rows.append(
        dict(
            class_id=cid,
            class_name=cname,
            precision=p_mask,
            sensitivity=r_mask,
            f1=F_mask,
            map50=ap50_m,
            map=ap_m,
        )
    )

df_classes = pd.DataFrame(per_class_rows)
print("\n=== Per-class FSP (mask-based) ===")
print(df_classes[["class_id", "class_name", "precision", "sensitivity", "f1"]])

# Save per-class metrics to results folder
df_classes.to_csv(RESULTS_DIR / "per_class_fsp.csv", index=False)
print(f"\nSaved per-class FSP to {RESULTS_DIR/'per_class_fsp.csv'}")


In [ ]:
# -----------------------------
# Global F / S / P bar chart
# -----------------------------
plt.figure(figsize=(5, 4))
metrics_names = ["Precision (P)", "Sensitivity (S)", "F1-score (F)"]
metric_values = [P_global, S_global, F_global]

plt.bar(metrics_names, metric_values)
plt.ylim(0, 1.0)
for i, v in enumerate(metric_values):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
plt.ylabel("Score")
plt.title("Global F / S / P (YOLOv8 Segmentation)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "global_fsp_bar.png", dpi=300)
plt.show()

# -----------------------------
# Per-class F / S / P bar charts
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

x = np.arange(len(df_classes))
labels = df_classes["class_name"].tolist()

axes[0].bar(x, df_classes["precision"])
axes[0].set_title("Per-class Precision")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

axes[1].bar(x, df_classes["sensitivity"])
axes[1].set_title("Per-class Sensitivity")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

axes[2].bar(x, df_classes["f1"])
axes[2].set_title("Per-class F1-score")
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

for ax in axes:
    ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_class_fsp_bar.png", dpi=300)
plt.show()


In [ ]:
# -----------------------------
# Simple XAI: last Conv activation heatmap
# + side-by-side: Original / YOLO seg / Heatmap
# -----------------------------

inner_model = model.model.to(device)
inner_model.eval()

# Find last Conv2d
conv_layers = [m for m in inner_model.modules() if isinstance(m, nn.Conv2d)]
assert conv_layers, "No Conv2d layers found in YOLO model."
last_conv = conv_layers[-1]
print("Using last Conv2d layer for XAI:", last_conv)

feature_maps = {}

def hook_fn(module, input, output):
    feature_maps["value"] = output.detach()

hook_handle = last_conv.register_forward_hook(hook_fn)

# Pick one test image
test_images = sorted(TEST_IMG_DIR.glob("*.jpg"))
assert test_images, f"No .jpg images found in {TEST_IMG_DIR}"
img_path = test_images[0]
print("Explaining image:", img_path.name)

# Read image
bgr = cv2.imread(str(img_path))
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

# YOLO segmentation prediction
results = model(str(img_path), verbose=False)
seg_vis_bgr = results[0].plot()   # BGR with masks/boxes drawn
seg_vis_rgb = cv2.cvtColor(seg_vis_bgr, cv2.COLOR_BGR2RGB)

# Get activation from hook
act = feature_maps.get("value", None)
assert act is not None, "Hook did not capture any activations."
act = act[0]  # [C, H, W]
print("Activation shape:", act.shape)

# Build heatmap
heat = act.mean(dim=0).cpu().numpy()
heat = np.maximum(heat, 0)
heat -= heat.min()
if heat.max() > 0:
    heat /= heat.max()

H, W, _ = rgb.shape
heat_resized = cv2.resize(heat, (W, H))

# Save and show side-by-side figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb)
axes[0].set_title("Original MRI")
axes[0].axis("off")

axes[1].imshow(seg_vis_rgb)
axes[1].set_title("YOLO Segmentation")
axes[1].axis("off")

axes[2].imshow(rgb)
axes[2].imshow(heat_resized, cmap="jet", alpha=0.4)
axes[2].set_title("XAI Heatmap (Last Conv)")
axes[2].axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "side_by_side_xai.png", dpi=300)
plt.show()

hook_handle.remove()


In [ ]:
cm_src = RUN_DIR / "confusion_matrix.png"
cmn_src = RUN_DIR / "confusion_matrix_normalized.png"

for src in [cm_src, cmn_src]:
    if src.exists():
        shutil.copy(src, RESULTS_DIR / src.name)
        print("Copied:", src.name)

# Show normalized confusion matrix if present
if cmn_src.exists():
    img = plt.imread(str(RESULTS_DIR / "confusion_matrix_normalized.png"))
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Normalized Confusion Matrix (saved by Ultralytics)")
    plt.show()
else:
    print("Normalized confusion matrix not found in run directory.")


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = RESULTS_DIR / "report.pdf"
print("Creating PDF report at:", pdf_path)

with PdfPages(pdf_path) as pdf:

    # Page 1: Text & global metrics
    fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait
    ax.axis("off")
    ax.text(0.5, 0.9, "Brain Tumor Segmentation – YOLOv8 Report", ha="center", fontsize=16, weight="bold")
    ax.text(0.0, 0.8, "Global Metrics (FSP):", fontsize=12, weight="bold")
    ax.text(0.05, 0.75, f"Precision (P):   {P_global:.4f}", fontsize=11)
    ax.text(0.05, 0.72, f"Sensitivity (S): {S_global:.4f}", fontsize=11)
    ax.text(0.05, 0.69, f"F1-score (F):    {F_global:.4f}", fontsize=11)

    ax.text(0.0, 0.62, "Per-class FSP (mask-based):", fontsize=12, weight="bold")

    # small table
    table_data = [["Class", "Precision", "Sensitivity", "F1-score"]]
    for _, row in df_classes.iterrows():
        table_data.append([
            str(row["class_name"]),
            f"{row['precision']:.3f}",
            f"{row['sensitivity']:.3f}",
            f"{row['f1']:.3f}",
        ])
    table = ax.table(cellText=table_data, colLabels=None, loc="upper left", colWidths=[0.2, 0.2, 0.2, 0.2])
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.2)

    pdf.savefig(fig)
    plt.close(fig)

    # Page 2: Global FSP bar plot
    img = plt.imread(RESULTS_DIR / "global_fsp_bar.png")
    fig, ax = plt.subplots(figsize=(8.27, 11.69))
    ax.imshow(img)
    ax.axis("off")
    pdf.savefig(fig)
    plt.close(fig)

    # Page 3: Per-class FSP bar plots
    img = plt.imread(RESULTS_DIR / "per_class_fsp_bar.png")
    fig, ax = plt.subplots(figsize=(8.27, 11.69))
    ax.imshow(img)
    ax.axis("off")
    pdf.savefig(fig)
    plt.close(fig)

    # Page 4: Side-by-side XAI
    xai_img_path = RESULTS_DIR / "side_by_side_xai.png"
    if xai_img_path.exists():
        img = plt.imread(xai_img_path)
        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

    # Page 5: Confusion matrix (if available)
    cmn_path = RESULTS_DIR / "confusion_matrix_normalized.png"
    if cmn_path.exists():
        img = plt.imread(cmn_path)
        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.imshow(img)
        ax.axis("off")
        pdf.savefig(fig)
        plt.close(fig)

print("PDF report created.")


In [ ]:
!pip install python-pptx


In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt

ppt_path = RESULTS_DIR / "report_slides.pptx"
prs = Presentation()

# Slide 1: Title + global metrics
slide_layout = prs.slide_layouts[0]  # title slide
slide = prs.slides.add_slide(slide_layout)
slide.shapes.title.text = "Brain Tumor Segmentation – YOLOv8"
subtitle = slide.placeholders[1]
subtitle.text = (
    f"Precision (P): {P_global:.4f}\n"
    f"Sensitivity (S): {S_global:.4f}\n"
    f"F1-score (F): {F_global:.4f}"
)

# Slide 2: Global FSP bar chart
slide_layout = prs.slide_layouts[5]  # title only
slide = prs.slides.add_slide(slide_layout)
slide.shapes.title.text = "Global F / S / P"
left = Inches(1)
top = Inches(1.5)
pic = slide.shapes.add_picture(str(RESULTS_DIR / "global_fsp_bar.png"), left, top, width=Inches(8))

# Slide 3: Per-class bar chart
slide = prs.slides.add_slide(slide_layout)
slide.shapes.title.text = "Per-class F / S / P"
pic = slide.shapes.add_picture(str(RESULTS_DIR / "per_class_fsp_bar.png"), left, top, width=Inches(8))

# Slide 4: XAI side-by-side
xai_img_path = RESULTS_DIR / "side_by_side_xai.png"
if xai_img_path.exists():
    slide = prs.slides.add_slide(slide_layout)
    slide.shapes.title.text = "Side-by-side: Original / YOLO / XAI"
    pic = slide.shapes.add_picture(str(xai_img_path), left, top, width=Inches(8))

# Slide 5: Confusion matrix (if available)
cmn_path = RESULTS_DIR / "confusion_matrix_normalized.png"
if cmn_path.exists():
    slide = prs.slides.add_slide(slide_layout)
    slide.shapes.title.text = "Normalized Confusion Matrix"
    pic = slide.shapes.add_picture(str(cmn_path), left, top, width=Inches(8))

prs.save(str(ppt_path))
print("PowerPoint saved at:", ppt_path)


In [ ]:
# =============================
# YOLOv8-Seg Architecture Diagram (Graphviz)
# =============================

from graphviz import Digraph

dot = Digraph("YOLOv8-Seg", format="png")
dot.attr(rankdir="LR", size="12,6")

# --- Nodes ---
dot.node("input", "Input MRI\n+ Preprocessing", shape="box", style="filled", fillcolor="#E0E0E0")

dot.node("backbone", "Backbone\nEfficientRep + C2f + SPPF\nOutputs: P3, P4, P5", 
         shape="box", style="filled", fillcolor="#AED6F1")

dot.node("neck", "Neck (FPN + PAN)\nOutputs: F3, F4, F5", 
         shape="box", style="filled", fillcolor="#F9E79F")

dot.node("detect", "Detection Head\n(Objectness, Class, Box Regression)", 
         shape="box", style="filled", fillcolor="#ABEBC6")

dot.node("segment", "Segmentation Head\n(K Prototypes + Mask Coefficients)", 
         shape="box", style="filled", fillcolor="#D7BDE2")

dot.node("boxes", "Final Bounding Boxes\n+ Tumor Labels", shape="box")
dot.node("masks", "Final Instance Masks\n(Segmentation)", shape="box")

# --- Edges ---
dot.edge("input", "backbone")
dot.edge("backbone", "neck")
dot.edge("neck", "detect")
dot.edge("neck", "segment")

dot.edge("detect", "boxes")
dot.edge("segment", "masks")

# Render
output_path = "/mnt/data/YOLOv8-Seg_Diagram"
dot.render(output_path, cleanup=True)

output_path + ".png"
